# ShopSphere E7T: re-tune E7 for its new features, then 5 seeds (GPU)

E7 kept E4T's settings, which were tuned before the 12 profile and region columns existed. This
notebook re-tunes them **with** those columns, then trains the tuned model (E7T) with 5 seeds.

- **Tuning:** Optuna, 30 trials, fold 0 only (as for E4T). **Trial 0 is E7's current settings**, so the
  search can only keep or beat them on fold 0. The search space is widened where E4T's best settings
  sat on its edges (hidden size, events per customer, dropout).
- **Decision rule, fixed before the results:** E7T replaces E7 only if, with 5 seeds each on all 64,000
  customers, it **wins on PR-AUC or ROC-AUC and loses on neither**, and the win also holds on folds 1-4
  (fold 0 chose the settings). The team's 80/20 holdout is checked once, after that decision.
  Never pick settings or seeds by their holdout score.

Inputs: **`shopsphere-data`**, **`shopsphere-code`** (newest version) and, for the final comparison,
**the output of `ShopSphere E7`** after its 5 seeds (its `oof_E7_seed*.csv`). Without it, the comparison
falls back to E4T.

Settings: **GPU T4**, **Save Version → Save & Run All**. About 6-8 hours (tuning stops starting new
trials after 3.5 hours). If the session ends early, attach this notebook's output as an input and run
again: finished trials, checkpoints and seeds are picked up.

In [ ]:
import glob, json, os, sys, subprocess, shutil, zipfile
import pandas as pd

def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    if not hits:
        for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):      # in case Kaggle kept a zip
            zipfile.ZipFile(z).extractall('/kaggle/working/unzipped')
        hits = sorted(glob.glob(f'/kaggle/working/unzipped/**/{pattern}', recursive=True))
    assert hits, f'{pattern} not found: attach the datasets to this notebook'
    return hits

DATA = os.path.dirname(find('shopsphere.db')[0])
# The code dataset may hold more than one version's folder: use the one whose tune.py has the E7T options.
code_dirs = [os.path.dirname(os.path.dirname(os.path.dirname(t))) for t in find('shopsphere_dl/tune.py')]
print('code folders found:', code_dirs)
code_dirs = [c for c in code_dirs if '--tag' in open(os.path.join(c, 'src', 'shopsphere_dl', 'tune.py')).read()]
assert code_dirs, 'only old code found: upload shopsphere_code_e7t.zip as a new version of shopsphere-code and attach it'
CODE = code_dirs[0]
print('data:', DATA); print('code:', CODE)

os.environ.update(SHOPSPHERE_DB=os.path.join(DATA, 'shopsphere.db'),
                  SHOPSPHERE_FOLDS=os.path.join(DATA, 'folds.csv'),
                  SHOPSPHERE_OUT='/kaggle/working',
                  PYTHONPATH=os.path.join(CODE, 'src'))
sys.path.insert(0, os.path.join(CODE, 'src'))

import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (turn on GPU T4)')

# Resume: files from an earlier version of THIS notebook attached as an input.
for pattern in ['tune_study_E7T.db', 'best_params_E7T.json', 'tune_trials_E7T.csv',
                'ckpt_E7T_*.pt', 'oof_E7T_seed*.csv', 'history_E7T_*.json']:
    for f in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True):
        dst = os.path.join('/kaggle/working', os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst); print('from earlier version:', os.path.basename(f))

# Reference: E7's out-of-fold scores from the ShopSphere E7 notebook's output, if attached.
os.makedirs('/kaggle/working/ref_e7', exist_ok=True)
for f in glob.glob('/kaggle/input/**/oof_E7_seed*.csv', recursive=True):
    shutil.copy(f, '/kaggle/working/ref_e7/')
E7_SEEDS = sorted(os.listdir('/kaggle/working/ref_e7'))
print('E7 reference files:', E7_SEEDS or 'none (comparison will use E4T)')

## 1. Tests. Training starts only if all pass.

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', os.path.join(CODE, 'tests')],
                   capture_output=True, text=True, cwd='/kaggle/working')
print(r.stdout[-3000:], r.stderr[-1500:])
assert r.returncode == 0, 'tests failed: do not train until they pass'

## 2. Tune on fold 0 (30 trials, at most 3.5 hours of new trials)

One line per epoch per trial. Pruned trials stop early. Writes `best_params_E7T.json` and
`tune_trials_E7T.csv`.

In [ ]:
TRIALS, HOURS = 30, 3.5

def stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd='/kaggle/working')
    for line in p.stdout:
        if 'Warning' not in line:
            print(line, end='')
    assert p.wait() == 0, 'failed: ' + ' '.join(cmd[2:5])

stream([sys.executable, '-m', 'shopsphere_dl.tune', '--extra', 'crm,geo', '--tag', 'E7T',
        '--start', os.path.join(CODE, 'best_params.json'), '--space', 'wide',
        '--trials', str(TRIALS), '--hours', str(HOURS), '--num-workers', '2'])

best = json.load(open('/kaggle/working/best_params_E7T.json'))
print('\nE7 settings on fold 0 (trial 0): PR-AUC', best['starting_settings_pr_auc_fold'])
print('best trial', best['trial'], 'PR-AUC', round(best['valid_pr_auc_fold'], 4), best['params'])
pd.read_csv('/kaggle/working/tune_trials_E7T.csv').sort_values('value', ascending=False).head(10)

## 3. Train E7T with the tuned settings, 5 seeds

If the best trial is trial 0 (E7's own settings), tuning found nothing better and E7T would just repeat
E7: the cell says so and stops.

In [ ]:
SEEDS = [1, 2, 3, 4, 5]

if best['trial'] == 0:
    print('Tuning kept E7\'s settings: nothing to retrain. Keep E7.')
else:
    for seed in SEEDS:
        if os.path.exists(f'/kaggle/working/oof_E7T_seed{seed}.csv'):
            print(f'seed {seed}: already done'); continue
        stream([sys.executable, '-m', 'shopsphere_dl.train', '--run', 'E4', '--seed', str(seed),
                '--params', '/kaggle/working/best_params_E7T.json', '--name', 'E7T', '--extra', 'crm,geo'])

## 4. Compare E7T with E7 (same seed count) and the baselines; re-check bias

Includes the folds 1-4 check (no tuning fold) and the team's 80/20 holdout with the team's rule.

In [ ]:
if glob.glob('/kaggle/working/oof_E7T_seed*.csv'):
    ref = ['--ref-run', 'E7', '--ref-run-dir', '/kaggle/working/ref_e7'] if len(E7_SEEDS) >= 5 else []
    if not ref:
        print('E7 reference not attached with 5 seeds: comparing with E4T instead.')
    r = subprocess.run([sys.executable, os.path.join(CODE, 'scripts', 'compare_e7.py'), '--new-dir', '/kaggle/working',
                        '--new-run', 'E7T', '--ref-dir', os.path.join(CODE, 'oof'), '--out', '/kaggle/working'] + ref,
                       capture_output=True, text=True, cwd='/kaggle/working')
    print(r.stdout, r.stderr[-2000:])
    assert r.returncode == 0

Download from the Output tab: `best_params_E7T.json`, `tune_trials_E7T.csv`, `oof_E7T_seed*.csv`,
`compare_5fold_e7t.csv`, `compare_holdout_e7t.csv`, `bootstrap_e7t.csv`, `bias_e7t.csv` and the
`history_E7T_*.json` files. Keep the `ckpt_E7T_*.pt` files in the output for the export step.